# Cleanup Target Import Volume

Removes **all contents** of the target import volume. Does not delete registered models. Use before re-running migration or to free space.

## Configuration

Read widget parameters: target catalog, schema, and import volume name. Cleans the entire volume root.

In [ ]:
dbutils.widgets.text("target_catalog", "", "Target catalog")
dbutils.widgets.text("schema", "default", "Schema")
dbutils.widgets.text("import_volume", "model_imports", "Import volume name")

TARGET_CATALOG = dbutils.widgets.get("target_catalog").strip()
SCHEMA = dbutils.widgets.get("schema").strip()
IMPORT_VOLUME = dbutils.widgets.get("import_volume").strip()

if not TARGET_CATALOG:
    raise ValueError("Set target_catalog")

VOLUME_PATH = f"/Volumes/{TARGET_CATALOG}/{SCHEMA}/{IMPORT_VOLUME}"
print(f"Cleaning entire volume (models NOT deleted): {VOLUME_PATH}")

## Remove import volume files

Recursively deletes all files and directories in the target import volume. Skips gracefully if the volume is empty or path does not exist.

In [ ]:
def path_not_found(e):
    msg = str(e).lower()
    return "path does not exist" in msg or "cannot find" in msg or "no such file" in msg or "filenotfound" in msg

try:
    entries = dbutils.fs.ls(VOLUME_PATH)
    for entry in entries:
        dbutils.fs.rm(entry.path, recurse=True)
        print(f"  Removed: {entry.path}")
    print(f"Cleaned: {VOLUME_PATH} ({len(entries)} items)")
except Exception as e:
    if path_not_found(e):
        print(f"Skip (not found): {VOLUME_PATH}")
    else:
        raise
print("Target volume cleanup done.")